# 1. Environment Set Up

In [ ]:
!apt-get update -y && apt-get install -y fuse lsb-release; \
    curl -L -O https://github.com/GoogleCloudPlatform/gcsfuse/releases/download/v0.41.12/gcsfuse_0.41.12_amd64.deb; \
    dpkg --install gcsfuse_0.41.12_amd64.deb; \
    apt-get update; \
    apt-get clean;

from google.colab import auth
auth.authenticate_user()

project_id = 'sfsu-378805'
bucket_name = 'csc-509-image-files'
!gcloud config set project {project_id}

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,788 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,243 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Pack

In [ ]:
# Mount the GCS bucket
!mkdir images
!gcsfuse --implicit-dirs {bucket_name} images

# 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 3. Imports

In [ ]:
# import packages
import numpy as np
import os
from pathlib import Path
import glob #access multiple files at a time, better to use glob when there are multiple of the same files (ex. nifti files) processing in batches
import pandas as pd
import math
import shutil
import random

# 4. Define Paths

In [ ]:
DATA_PATH = Path('/content/images/Module1_BraTS')
train_data_path = DATA_PATH / 'MICCAI_BraTS2020_TrainingData'
valid_data_path = DATA_PATH / 'MICCAI_BraTS2020_ValidationData'

# 5. Train/Val/Test Split

In [ ]:
random.seed(42)

def get_patient_ids(path):
    return [x for x in os.listdir(path) if not x.endswith('.nii.gz') and '.csv' not in x]

train_patients = get_patient_ids(train_data_path)
valid_patients = get_patient_ids(valid_data_path)

all_patients = train_patients + valid_patients
random.shuffle(all_patients)

In [ ]:
# 80/10/10 split
num_total = len(all_patients)
num_train = math.floor(0.8 * num_total)
num_val = math.floor(0.1 * num_total)

train_split = all_patients[:num_train]
val_split = all_patients[num_train:num_train + num_val]
test_split = all_patients[num_train + num_val:]

# 6. Save Split Log CSV to Drive

In [ ]:
def get_source_dir(pid):
    return 'MICCAI_BraTS2020_TrainingData' if 'Training' in pid else 'MICCAI_BraTS2020_ValidationData'

In [ ]:
split_log = []
for pid in train_split:
    split_log.append(['train', pid, get_source_dir(pid)])
for pid in val_split:
    split_log.append(['val', pid, get_source_dir(pid)])
for pid in test_split:
    split_log.append(['test', pid, get_source_dir(pid)])

split_log_df = pd.DataFrame(split_log, columns=['new_split', 'patient_id', 'source_dir'])
split_log_df.to_csv('/content/drive/MyDrive/split_log.csv', index=False)


# 7. Load Split Log in Future Sessions

In [ ]:
split_log_df = pd.read_csv('/content/drive/MyDrive/split_log.csv')

# 8. Get patient paths for a split

In [ ]:
def get_patient_paths(split_csv_path, split_name):
    df = pd.read_csv(split_csv_path)
    df = df[df['new_split'] == split_name]
    base_path = Path('/content/images/Module1_BraTS')
    paths = []
    for _, row in df.iterrows():
        patient_path = base_path / row['source_dir'] / row['patient_id']
        paths.append(patient_path)
    return paths


In [ ]:
# Example usage:
train_paths = get_patient_paths('/content/drive/MyDrive/split_log.csv', 'train')
print(f"Loaded {len(train_paths)} training patients")



---



#. Evaluation Metrics

In [ ]:
# # semantic segmentation
# def dice_coef(y_true, y_pred, smooth=1e-6):
#     """
#     Dice coefficient for semantic segmentation.

#     Dice = (2*|X & Y|)/ (|X|+ |Y|)
#          =  2*sum(|A*B|)/(sum(A^2)+sum(B^2))

#     y_true: The ground truth tensor.
#     y_pred: The predicted tensor.
#     smooth: Smoothing factor to avoid division by zero.
#     """
#     y_true_f = y_true.view(-1)
#     y_pred_f = y_pred.view(-1)
#     intersection = (y_true_f * y_pred_f).sum()
#     return (2. * intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() + smooth)

# def jaccard_coef(y_true, y_pred, smooth=1e-6):
#     """
#     Jaccard coefficient for semantic segmentation. Also known as the IoU loss.

#     Jaccard = (|X & Y|)/ (|X|+ |Y| - |X & Y|)
#             = sum(|A*B|)/(sum(|A|)+sum(|B|)-sum(|A*B|))

#     y_true: The ground truth tensor.
#     y_pred: The predicted tensor.
#     smooth: Smoothing factor to avoid division by zero.
#     """
#     y_true_f = y_true.view(-1)
#     y_pred_f = y_pred.view(-1)
#     intersection = (y_true_f * y_pred_f).sum()
#     return (intersection + smooth) / (y_true_f.sum() + y_pred_f.sum() - intersection + smooth)

In [ ]:
# # loss functions for semantic segmentation
# def dice_coef_loss(y_true, y_pred):
#     """
#     Dice loss for semantic segmentation.
#     Minimizing this loss corresponds to maximizing the Dice coefficient.
#     """
#     return 1 - dice_coef(y_true, y_pred)

# def jaccard_coef_loss(y_true, y_pred):
#     """
#     Jaccard loss for semantic segmentation.
#     Minimizing this loss corresponds to maximizing the Jaccard coefficient.
#     """
#     return 1 - jaccard_coef(y_true, y_pred)